In [ ]:
# Install Whisper and FFMPEG
!pip install git+https://github.com/openai/whisper.git
!sudo apt update && sudo apt install ffmpeg

# Import necessary libraries
from multiprocessing import Pool
import subprocess
import os

# Define the directory containing the audio files
audio_dir = "audio_files"

# Check if the directory exists, create it if not
if not os.path.exists(audio_dir):
    os.makedirs(audio_dir)
    print(f"Created directory '{audio_dir}'. Please upload your MP3 files to this directory using the file browser on the left.")
else:
    print(f"Directory '{audio_dir}' already exists.")

# Define the transcribe function with improved formatting options
def transcribe(filename):
    try:
        # This is the key part of the solution
        # We've added --word_timestamps, --max_line_width, and --max_line_count
        subprocess.run([
            "whisper",
            filename,
            "--model", "turbo",
            "--language", "en",
            "--output_format", "srt",
            "--output_dir", "transcripts",
            "--word_timestamps", "True",   # Added: Gets timing for each word for more precise cuts
            "--max_line_width", "42",      # Added: Ensures lines don't get too long (standard is ~42 chars)
            "--max_line_count", "2"        # Added: Ensures no more than 2 lines per subtitle block
        ], check=True)
        print(f"Successfully transcribed {os.path.basename(filename)}")
    except subprocess.CalledProcessError as e:
        print(f"Error transcribing {filename}: {e}")

In [ ]:
# Get a list of all MP3 files in the directory
try:
    filenames = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir) if f.endswith('.mp3')]
    if not filenames:
        print(f"No MP3 files found in '{audio_dir}'. Please upload some and re-run this cell.")
    else:
        print(f"Found {len(filenames)} MP3 files to transcribe.")
except FileNotFoundError:
    print(f"The directory '{audio_dir}' does not exist.")
    filenames = []

# Main block to transcribe files in parallel
if __name__ == '__main__':
    # Only run if there are files to process
    if filenames:
        # Adjust the number of processes based on your CPU cores (e.g., 2, 4)
        with Pool(processes=2) as pool:
            pool.map(transcribe, filenames)
        print("\nAll transcriptions complete!")
    else:
        print("No files to transcribe.")

In [ ]:
# Zip the transcription outputs for download
if os.path.exists("transcripts") and os.listdir("transcripts"):
    !zip -r transcripts.zip transcripts/
    print("\nTranscripts have been zipped into 'transcripts.zip' for download.")
else:
    print("\nNo transcripts folder found or it is empty. Cannot create zip file.")